# two-optimizers-alternating-step — faded example 1: Detach the fake in the D-step

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `two-optimizers-alternating-step`. Running the beacon reports progress on the `GAN: Two-optimizers alternating step` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: Two-optimizers alternating step` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`two-optimizers-alternating-step`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "two-optimizers-alternating-step"
DD_SUBTOPIC = "GAN: Two-optimizers alternating step"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

During the discriminator step the generator must be frozen: the fake batch is `G(z).detach()` so backprop of the D loss stops at the fake and never updates `G`. Forgetting the detach silently lets the D-step's gradient leak into the generator.

## Faded exercise 1

### Build the discriminator's fake input

Implement `d_step(G, D, D_opt, z, x_real)`: one discriminator update of the GAN loop returning the D loss as a float. Everything is wired except the construction of the fake batch — complete it so the generator is frozen during this step.

**Fill in:** the detached generator output G(z).detach() used as the fake batch

In [ ]:
import torch as t
import torch.nn as nn

def d_step(G, D, D_opt, z, x_real):
    D_opt.zero_grad()
    fake = None  # TODO: the detached generator output G(z).detach() used as the fake batch
    loss_D = (D(fake) - D(x_real)).mean()
    loss_D.backward()
    D_opt.step()
    return loss_D.item()

t.manual_seed(0)
G = nn.Linear(3, 4); D = nn.Linear(4, 1)
D_opt = t.optim.SGD(D.parameters(), lr=0.01)
print(d_step(G, D, D_opt, t.randn(5, 3), t.randn(5, 4)))


def _test():
    t.manual_seed(0)
    G = nn.Linear(3, 4); D = nn.Linear(4, 1)
    D_opt = t.optim.SGD(D.parameters(), lr=0.01)
    z = t.randn(5, 3); x_real = t.randn(5, 4)
    G_before = G.weight.detach().clone()
    loss = d_step(G, D, D_opt, z, x_real)
    assert isinstance(loss, float)
    # the detach must keep the generator frozen: G unchanged by a D-step
    assert t.equal(G_before, G.weight.detach()), 'generator must not move during D-step'
    # and the generator received no gradient at all
    assert G.weight.grad is None or float(G.weight.grad.abs().sum()) == 0.0


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

def d_step(G, D, D_opt, z, x_real):
    D_opt.zero_grad()
    fake = G(z).detach()
    loss_D = (D(fake) - D(x_real)).mean()
    loss_D.backward()
    D_opt.step()
    return loss_D.item()

t.manual_seed(0)
G = nn.Linear(3, 4); D = nn.Linear(4, 1)
D_opt = t.optim.SGD(D.parameters(), lr=0.01)
print(d_step(G, D, D_opt, t.randn(5, 3), t.randn(5, 4)))
```
</details>